# 01 - Feature Engineering

**Goal:** Build feature matrices (one per position) that every subsequent modelling phase depends on.

## 1. Database Connection

In [1]:
import sys
import os

# Add project root to path so we can import src modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from src.pg_config import get_pg_config

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Build SQLAlchemy engine from existing project config
cfg = get_pg_config()
engine = create_engine(
    f'postgresql+psycopg2://{cfg.user}:{cfg.password}@{cfg.host}:{cfg.port}/{cfg.dbname}'
)
print(f'Connected to {cfg.dbname} on {cfg.host}:{cfg.port}')

Connected to fpl_db on localhost:7651


## 2. Load Raw Data from Gold Schema

In [2]:
# Primary data source: per-player, per-gameweek stats
history = pd.read_sql('SELECT * FROM gold.players_history ORDER BY element, round', engine)

# Player metadata (need element_type for position)
players = pd.read_sql('SELECT * FROM gold.players', engine)

# Team strength ratings
teams = pd.read_sql('SELECT * FROM gold.teams', engine)

# Fixture difficulty
fixtures = pd.read_sql('SELECT * FROM gold.fixtures_main', engine)

print(f'history:  {history.shape[0]:,} rows x {history.shape[1]} cols')
print(f'players:  {players.shape[0]:,} rows x {players.shape[1]} cols')
print(f'teams:    {teams.shape[0]:,} rows x {teams.shape[1]} cols')
print(f'fixtures: {fixtures.shape[0]:,} rows x {fixtures.shape[1]} cols')

history:  23,829 rows x 41 cols
players:  825 rows x 88 cols
teams:    20 rows x 13 cols
fixtures: 380 rows x 12 cols


In [3]:
# Quick look at the history table — this is our primary dataset
history.head(3)

,element,fixture,opponent_team,total_points,was_home,kickoff_time,team_h_score,team_a_score,round,modified,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,penalties_missed,yellow_cards,red_cards,saves,bonus,bps,influence,creativity,threat,ict_index,clearances_blocks_interceptions,recoveries,tackles,defensive_contribution,starts,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,value,transfers_balance,selected,transfers_in,transfers_out
0,1,9,14,10,False,2025-08-17 15:30:00,0,1,1,False,90,0,0,1,0,0,0,0,1,0,7,3,38,49.2,0.0,0.0,4.9,1,13,0,0,1,0.0,0.00,0.00,1.52,55,0,1531911,0,0
1,1,11,11,6,True,2025-08-23 16:30:00,5,0,2,False,90,0,0,1,0,0,0,0,0,0,1,0,28,13.4,0.0,0.0,1.3,0,3,0,0,1,0.0,0.00,0.00,0.17,55,218659,2284634,277339,58680
2,1,25,12,2,False,2025-08-31 15:30:00,1,0,3,False,90,0,0,0,1,0,0,0,0,0,2,0,12,20.0,10.0,0.0,3.0,0,12,0,0,1,0.0,0.02,0.02,0.52,55,-12311,2406964,146739,159050


In [4]:
history.dtypes

element                                     int64
fixture                                     int64
opponent_team                               int64
total_points                                int64
was_home                                     bool
kickoff_time                       datetime64[ns]
team_h_score                                int64
team_a_score                                int64
round                                       int64
modified                                     bool
minutes                                     int64
goals_scored                                int64
assists                                     int64
clean_sheets                                int64
goals_conceded                              int64
own_goals                                   int64
penalties_saved                             int64
penalties_missed                            int64
yellow_cards                                int64
red_cards                                   int64


In [5]:
# How many unique players and gameweeks?
print(f'Unique players: {history["element"].nunique()}')
print(f'Gameweek range: {history["round"].min()} - {history["round"].max()}')
print(f'Rows with 0 minutes: {(history["minutes"] == 0).sum()} ({(history["minutes"] == 0).mean():.1%})')

Unique players: 825
Gameweek range: 1 - 31
Rows with 0 minutes: 14484 (60.8%)


**Important design decision - data for players with 0 minutes played is not filtered from history table - we keep those records at least until the rolling stats are calculated.**

## 3. Rolling Form Features (3 and 5 GW averages)

These capture *recent* performance. A player's last 3 GW average is more predictive than their season average because form fluctuates.

Method: `groupby('element').shift(1).rolling(N).mean()`
- `groupby('element')` — per player
- `.shift(1)` — exclude current GW (prevents leakage)
- `.rolling(N).mean()` — average of last N values

In [6]:
# Columns to compute rolling averages for
rolling_cols = [
    'total_points', 'minutes', 'goals_scored', 'assists',
    'expected_goals', 'expected_assists', 'expected_goal_involvements',
    'ict_index', 'clean_sheets', 'goals_conceded',
    'expected_goals_conceded', 'bonus', 'influence', 'creativity', 'threat'
]

df = history

for col in rolling_cols:
    # Cast to float to handle potential integer columns with NaN
    shifted = df.groupby('element')[col].shift(1)
    
    for window in [3, 5]:
        short_name = col.replace('expected_goals', 'xg') \
                       .replace('expected_assists', 'xa') \
                       .replace('expected_goal_involvements', 'xgi') \
                       .replace('expected_goals_conceded', 'xgc') \
                       .replace('total_points', 'pts') \
                       .replace('goals_scored', 'goals') \
                       .replace('clean_sheets', 'cs') \
                       .replace('goals_conceded', 'gc') \
                       .replace('ict_index', 'ict')
        df[f'{short_name}_rolling_{window}'] = shifted.rolling(window, min_periods=1).mean()

print(f'Added {len(rolling_cols) * 2} rolling mean features')

Added 30 rolling mean features


In [7]:
df

,element,fixture,opponent_team,total_points,was_home,kickoff_time,team_h_score,team_a_score,round,modified,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,penalties_missed,yellow_cards,red_cards,saves,bonus,bps,influence,creativity,threat,ict_index,clearances_blocks_interceptions,recoveries,tackles,...,pts_rolling_3,pts_rolling_5,minutes_rolling_3,minutes_rolling_5,goals_rolling_3,goals_rolling_5,assists_rolling_3,assists_rolling_5,xg_rolling_3,xg_rolling_5,xa_rolling_3,xa_rolling_5,xgi_rolling_3,xgi_rolling_5,ict_rolling_3,ict_rolling_5,cs_rolling_3,cs_rolling_5,gc_rolling_3,gc_rolling_5,xg_conceded_rolling_3,xg_conceded_rolling_5,bonus_rolling_3,bonus_rolling_5,influence_rolling_3,influence_rolling_5,creativity_rolling_3,creativity_rolling_5,threat_rolling_3,threat_rolling_5
0,1,9,14,10,False,2025-08-17 15:30:00,0,1,1,False,90,0,0,1,0,0,0,0,1,0,7,3,38,49.2,0.0,0.0,4.9,1,13,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,11,11,6,True,2025-08-23 16:30:00,5,0,2,False,90,0,0,1,0,0,0,0,0,0,1,0,28,13.4,0.0,0.0,1.3,0,3,0,...,10.000000,10.0,90.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,4.900000,4.900000,1.000000,1.000000,0.000000,0.000000,1.520000,1.520000,3.0,3.00,49.200000,49.200000,0.000000,0.000000,0.0,0.0
2,1,25,12,2,False,2025-08-31 15:30:00,1,0,3,False,90,0,0,0,1,0,0,0,0,0,2,0,12,20.0,10.0,0.0,3.0,0,12,0,...,8.000000,8.0,90.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,3.100000,3.100000,1.000000,1.000000,0.000000,0.000000,0.845000,0.845000,1.5,1.50,31.300000,31.300000,0.000000,0.000000,0.0,0.0
3,1,31,16,6,True,2025-09-13 11:30:00,3,0,4,False,90,0,0,1,0,0,0,0,0,0,1,0,24,12.8,0.0,0.0,1.3,0,9,0,...,6.000000,6.0,90.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006667,0.006667,0.006667,0.006667,3.066667,3.066667,0.666667,0.666667,0.333333,0.333333,0.736667,0.736667,1.0,1.00,27.533333,27.533333,3.333333,3.333333,0.0,0.0
4,1,41,13,2,True,2025-09-21 15:30:00,1,1,5,False,90,0,0,0,1,0,0,0,0,0,2,0,13,21.4,0.0,0.0,2.1,1,6,0,...,4.666667,6.0,90.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006667,0.005000,0.006667,0.005000,1.866667,2.625000,0.666667,0.750000,0.333333,0.250000,0.296667,0.602500,0.0,0.75,15.400000,23.850000,3.333333,2.500000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23824,822,299,6,0,True,2026-03-14 15:00:00,0,1,30,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0
23825,822,308,15,0,False,2026-03-22 12:00:00,1,2,31,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0
23826,823,306,11,0,False,2026-03-21 20:00:00,0,0,31,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0
23827,824,306,11,0,False,2026-03-21 20:00:00,0,0,31,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0


## 4. Rolling Standard Deviation

How *consistent* is the player? A player averaging 5 pts with std=1 is more reliable than one averaging 5 with std=4.

In [8]:
std_cols = ['total_points', 'minutes', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded']

for col in std_cols:
    shifted = df.groupby('element')[col].shift(1)
    for window in [3, 5]:
        short_name = col.replace('total_points', 'pts') \
                        .replace('expected_goals', 'xg') \
                        .replace('expected_assists', 'xa') \
                        .replace('expected_goal_involvements', 'xgi') \
                        .replace('expected_goals_conceded', 'xgc')

        df[f'{short_name}_std_{window}'] = shifted.rolling(window, min_periods=1).std()

print(f'Added {len(std_cols) * 2} rolling std features')
[c for c in df.columns if '_std_' in c]

Added 12 rolling std features


['pts_std_3',
 'pts_std_5',
 'minutes_std_3',
 'minutes_std_5',
 'xg_std_3',
 'xg_std_5',
 'xa_std_3',
 'xa_std_5',
 'xgi_std_3',
 'xgi_std_5',
 'xg_conceded_std_3',
 'xg_conceded_std_5']

## 5. Season Aggregates

These capture the *baseline quality* of a player, separate from recent form.
- `season_avg_pts` — expanding mean (all prior GWs, excluding current)
- `season_total_minutes` — cumulative minutes played

In [9]:
# Expanding mean of points (shift first to exclude current GW)
df['season_avg_pts'] = df.groupby('element')['total_points'].apply(
    lambda x: x.shift(1).expanding().mean()
).reset_index(level=0, drop=True)

# Cumulative minutes (shift so it reflects minutes BEFORE this GW)
df['season_total_minutes'] = df.groupby('element')['minutes'].apply(
    lambda x: x.shift(1).expanding().sum()
).reset_index(level=0, drop=True)

print('Season aggregates added')
df[['element', 'round', 'total_points', 'minutes', 'season_avg_pts', 'season_total_minutes']].head(10)

Season aggregates added


,element,round,total_points,minutes,season_avg_pts,season_total_minutes
0,1,1,10,90,NaN,NaN
1,1,2,6,90,10.000000,90.0
2,1,3,2,90,8.000000,180.0
3,1,4,6,90,6.000000,270.0
4,1,5,2,90,6.000000,360.0
5,1,6,2,90,5.200000,450.0
6,1,7,6,90,4.666667,540.0
7,1,8,6,90,4.857143,630.0
8,1,9,6,90,5.000000,720.0
9,1,10,6,90,5.111111,810.0


## 6. Opponent Strength

Before each match, we know who the opponent is. Teams have different home/away strength ratings.

For a player:
- If they're playing **at home**, their opponent is the *away* team → use opponent's **away** attack/defence strength
- If they're playing **away**, their opponent is the *home* team → use opponent's **home** attack/defence strength

In [10]:
# Build opponent strength lookup
# When opponent is playing away (player is home): use opponent's away strengths
# When opponent is playing home (player is away): use opponent's home strengths

teams_cols = teams[['id', 'strength', 'strength_overall_home', 'strength_overall_away',
                     'strength_attack_home', 'strength_attack_away',
                     'strength_defence_home', 'strength_defence_away']].copy()

# For rows where player is HOME → opponent is AWAY
home_merge = teams_cols.rename(columns={
    'id': 'opponent_team',
    'strength': 'opp_strength',
    'strength_overall_away': 'opp_strength_overall',
    'strength_attack_away': 'opp_strength_attack',
    'strength_defence_away': 'opp_strength_defence'
})[['opponent_team', 'opp_strength', 'opp_strength_overall', 'opp_strength_attack', 'opp_strength_defence']]

# For rows where player is AWAY → opponent is HOME
away_merge = teams_cols.rename(columns={
    'id': 'opponent_team',
    'strength': 'opp_strength',
    'strength_overall_home': 'opp_strength_overall',
    'strength_attack_home': 'opp_strength_attack',
    'strength_defence_home': 'opp_strength_defence'
})[['opponent_team', 'opp_strength', 'opp_strength_overall', 'opp_strength_attack', 'opp_strength_defence']]

# Split, merge, recombine
df_home = df[df['was_home'] == True].merge(home_merge, on='opponent_team', how='left')
df_away = df[df['was_home'] == False].merge(away_merge, on='opponent_team', how='left')
df = pd.concat([df_home, df_away]).sort_values(['element', 'round']).reset_index(drop=True)

print('Opponent strength features added')
df[['element', 'round', 'opponent_team', 'was_home', 'opp_strength_overall', 'opp_strength_attack', 'opp_strength_defence']].head(6)

Opponent strength features added


,element,round,opponent_team,was_home,opp_strength_overall,opp_strength_attack,opp_strength_defence
0,1,1,14,False,1180,1100,1260
1,1,2,11,True,1200,1140,1260
2,1,3,12,False,1215,1170,1260
3,1,4,16,True,1145,1120,1170
4,1,5,13,True,1350,1300,1400
5,1,6,15,False,1150,1150,1150


## 7. FDR

- `fixture_difficulty` — FDR rating (1-5) from fixtures table. 1 = easy, 5 = hardest.

In [11]:
# is_home: convert boolean to int for the model
df['is_home'] = df['was_home'].astype(int)

# Fixture difficulty: need to figure out which team the player belongs to
# fixtures_main has team_h, team_a, team_h_difficulty, team_a_difficulty
# For home players: use team_h_difficulty; for away players: use team_a_difficulty

# We need to join on fixture id
fixture_diff = fixtures[['id', 'team_h_difficulty', 'team_a_difficulty']].rename(
    columns={'id': 'fixture'}
)

df = df.merge(fixture_diff, on='fixture', how='left')

# Select the right difficulty based on whether player was home or away
df['fixture_difficulty'] = np.where(
    df['was_home'],
    df['team_h_difficulty'],
    df['team_a_difficulty']
)

# Drop the intermediate columns
df.drop(columns=['team_h_difficulty', 'team_a_difficulty', 'was_home'], inplace=True)

print('Fixture context added')
df[['element', 'round', 'is_home', 'fixture_difficulty']].head(6)

Fixture context added


,element,round,is_home,fixture_difficulty
0,1,1,0,4
1,1,2,1,2
2,1,3,0,4
3,1,4,1,2
4,1,5,1,4
5,1,6,0,4


## 8. Player Value Features

- `value` — FPL price this GW (already in history table, in tenths: 45 = 4.5m)
- `value_change_3gw` — price change over last 3 GWs (rising price = crowd thinks player is performing)

In [12]:
# value is already in the dataframe from players_history
# Compute 3GW value change (current value minus value 3 GWs ago)
df['value_change_3gw'] = df.groupby('element')['value'].diff(3)
df['value_change_3gw'] = df['value_change_3gw'].fillna(0)

print('Value features added')
df[['element', 'round', 'value', 'value_change_3gw']].head(8)

Value features added


,element,round,value,value_change_3gw
0,1,1,55,0.0
1,1,2,55,0.0
2,1,3,55,0.0
3,1,4,55,0.0
4,1,5,55,0.0
5,1,6,56,1.0
6,1,7,56,1.0
7,1,8,57,2.0


## 9. Add Position from Players Table

We need `element_type` to split into position-specific models. The players table has this mapped to human-readable names.

In [13]:
players

,ID,First Name,Second Name,chance_of_playing_next_round,chance_of_playing_this_round,code,cost_change_event,cost_change_event_fall,cost_change_start,cost_change_start_fall,dreamteam_count,Position,ep_next,ep_this,event_points,form,in_dreamteam,news,news_added,Price,Points Per Game,selected_by_percent,status,Team,Points,transfers_in,transfers_in_event,transfers_out,transfers_out_event,value_form,...,xG,xA,xGI,xGC,influence_rank,influence_rank_type,threat_rank,threat_rank_type,ict_index_rank_type,corners_and_indirect_freekicks_order,direct_freekicks_order,penalties_order,xG90,Saves90,xA90,xGI90,xGC90,GC90,now_cost_rank,now_cost_rank_type,form_rank,form_rank_type,points_per_game_rank_type,selected_rank,selected_rank_type,starts_per_90,CS90,DC90,Goals90,Assists90
0,1,David,Raya Martín,NaN,NaN,154561,0,0,5,-5,1,Goalkeeper,6.0,0.0,0,5.0,True,,NaT,6.0,4.2,34.5,a,Arsenal,129,4147153,26476,2472709,5415,0.8,...,0.0,0.06,0.06,22.30,108,18,816,95,16,NaN,NaN,NaN,0.00,1.61,0.00,0.00,0.72,0.71,86,1,36,3,3,9,1,1.0,0.48,0.00,0.00,0.00
1,2,Kepa,Arrizabalaga Revuelta,NaN,NaN,109745,0,0,-5,5,0,Goalkeeper,1.0,0.0,0,0.0,False,,NaT,4.0,0.0,0.4,a,Arsenal,0,14808,56,77494,85,0.0,...,0.0,0.00,0.00,0.00,570,66,529,38,66,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,668,43,465,59,66,285,37,0.0,0.00,0.00,0.00,0.00
2,3,Karl,Hein,0.0,0.0,463748,0,0,0,0,0,Goalkeeper,0.0,0.0,0,0.0,False,Has joined Werder Bremen on loan for the rest ...,2025-08-26 13:44:02.357864,4.0,0.0,0.2,u,Arsenal,0,5545,0,48328,30,0.0,...,0.0,0.00,0.00,0.00,587,75,548,49,75,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,689,54,483,69,75,373,54,0.0,0.00,0.00,0.00,0.00
3,4,Tommy,Setford,NaN,NaN,551221,0,0,-1,1,0,Goalkeeper,1.0,0.0,0,0.0,False,,NaT,3.9,0.0,0.2,a,Arsenal,0,33931,17,34191,89,0.0,...,0.0,0.00,0.00,0.00,559,60,515,30,60,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,780,90,453,52,60,362,52,0.0,0.00,0.00,0.00,0.00
4,5,Gabriel,dos Santos Magalhães,100.0,100.0,226597,0,0,12,-12,5,Defender,8.8,0.0,0,7.8,True,,2025-11-17 12:00:07.436586,7.2,6.9,43.1,a,Arsenal,173,8308673,45333,5388922,2398,1.1,...,1.9,1.65,3.55,15.82,26,9,142,27,25,NaN,NaN,NaN,0.08,0.00,0.07,0.15,0.66,0.62,32,1,2,1,1,5,1,1.0,0.58,9.31,0.12,0.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
820,821,Jack,Whittaker,NaN,NaN,570929,0,0,0,0,0,Midfielder,0.0,0.0,0,0.0,False,,NaT,4.5,0.0,0.0,a,Sunderland,0,30,10,3,2,0.0,...,0.0,0.00,0.00,0.00,728,298,709,294,299,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,473,289,663,258,301,821,369,0.0,0.00,0.00,0.00,0.00
821,822,Jaydon,Jones,NaN,NaN,571087,0,0,0,0,0,Midfielder,0.0,0.0,0,0.0,False,,NaT,4.5,0.0,0.0,a,Sunderland,0,46,19,16,5,0.0,...,0.0,0.00,0.00,0.00,729,299,710,295,300,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,474,290,664,259,302,820,368,0.0,0.00,0.00,0.00,0.00
822,823,Joshua,Stephenson,NaN,NaN,649240,0,0,0,0,0,Defender,1.4,1.4,0,0.0,False,,NaT,4.0,0.0,0.0,a,Brentford,0,3,3,0,0,0.0,...,0.0,0.00,0.00,0.00,628,248,592,246,248,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,741,213,527,218,248,824,268,0.0,0.00,0.00,0.00,0.00
823,824,Conor,McManus,NaN,NaN,587433,0,0,0,0,0,Defender,1.4,1.4,0,0.0,False,,NaT,4.0,0.0,0.0,a,Brentford,0,8,8,1,1,0.0,...,0.0,0.00,0.00,0.00,615,236,578,233,236,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,724,197,512,204,236,823,267,0.0,0.00,0.00,0.00,0.00


In [14]:
# The players table uses aliased column names — "ID" and "Position"
position_map = players[['ID', 'Position']].rename(columns={'ID': 'element'})
df = df.merge(position_map, on='element', how='left')

In [15]:
df['bps']

0        38
1        28
2        12
3        24
4        13
         ..
23824     0
23825     0
23826     0
23827     0
23828     0
Name: bps, Length: 23829, dtype: int64

## 10. Define Feature Set and Target

Now we select which columns are features (inputs to the model) and which is the target (what we predict).

In [16]:
TARGET = 'total_points'

# Identifiers (not features, but needed for debugging/splitting)
ID_COLS = ['element', 'round', 'Position', 'kickoff_time']

# Raw columns from history that should NOT be features
# (they are either the target, identifiers, or would cause leakage)
EXCLUDE_COLS = [
    'total_points',       # target
    'fixture',            # identifier
    'opponent_team',      # used to build opp_strength, not a feature itself
    'was_home',           # replaced by is_home
    'kickoff_time',       # used to build days_since_last_match
    'team_h_score',       # match result — leakage!
    'team_a_score',       # match result — leakage!
    'modified',           # metadata
    'element',            # identifier
    'round',              # will use for temporal split, not as feature
    'Position',           # used for splitting, not as feature within position models
    # Raw per-GW stats are also leakage if used as features
    # (they describe THIS gameweek's outcome, not prior info)
    'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded',
    'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards',
    'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity',
    'threat', 'ict_index', 'clearances_blocks_interceptions', 'recoveries',
    'tackles', 'defensive_contribution', 'starts',
    'expected_goals', 'expected_assists', 'expected_goal_involvements',
    'expected_goals_conceded',
    'transfers_balance', 'selected', 'transfers_in', 'transfers_out',
]

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f'Target: {TARGET}')
print(f'Features ({len(FEATURE_COLS)}):')
for i, col in enumerate(sorted(FEATURE_COLS)):
    print(f'  {i+1:2d}. {col}')

Target: total_points
Features (52):
   1. assists_rolling_3
   2. assists_rolling_5
   3. bonus_rolling_3
   4. bonus_rolling_5
   5. creativity_rolling_3
   6. creativity_rolling_5
   7. cs_rolling_3
   8. cs_rolling_5
   9. fixture_difficulty
  10. gc_rolling_3
  11. gc_rolling_5
  12. goals_rolling_3
  13. goals_rolling_5
  14. ict_rolling_3
  15. ict_rolling_5
  16. influence_rolling_3
  17. influence_rolling_5
  18. is_home
  19. minutes_rolling_3
  20. minutes_rolling_5
  21. minutes_std_3
  22. minutes_std_5
  23. opp_strength
  24. opp_strength_attack
  25. opp_strength_defence
  26. opp_strength_overall
  27. pts_rolling_3
  28. pts_rolling_5
  29. pts_std_3
  30. pts_std_5
  31. season_avg_pts
  32. season_total_minutes
  33. threat_rolling_3
  34. threat_rolling_5
  35. value
  36. value_change_3gw
  37. xa_rolling_3
  38. xa_rolling_5
  39. xa_std_3
  40. xa_std_5
  41. xg_conceded_rolling_3
  42. xg_conceded_rolling_5
  43. xg_conceded_std_3
  44. xg_conceded_std_5
  45. x

## 11. Inspect Nulls

Rolling features will be NaN for early gameweeks (not enough history). This is expected — those rows will be dropped.

In [17]:
df

,element,fixture,opponent_team,total_points,kickoff_time,team_h_score,team_a_score,round,modified,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,penalties_missed,yellow_cards,red_cards,saves,bonus,bps,influence,creativity,threat,ict_index,clearances_blocks_interceptions,recoveries,tackles,defensive_contribution,...,bonus_rolling_3,bonus_rolling_5,influence_rolling_3,influence_rolling_5,creativity_rolling_3,creativity_rolling_5,threat_rolling_3,threat_rolling_5,pts_std_3,pts_std_5,minutes_std_3,minutes_std_5,xg_std_3,xg_std_5,xa_std_3,xa_std_5,xgi_std_3,xgi_std_5,xg_conceded_std_3,xg_conceded_std_5,season_avg_pts,season_total_minutes,opp_strength,opp_strength_overall,opp_strength_attack,opp_strength_defence,is_home,fixture_difficulty,value_change_3gw,Position
0,1,9,14,10,2025-08-17 15:30:00,0,1,1,False,90,0,0,1,0,0,0,0,1,0,7,3,38,49.2,0.0,0.0,4.9,1,13,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,1180,1100,1260,0,4,0.0,Goalkeeper
1,1,11,11,6,2025-08-23 16:30:00,5,0,2,False,90,0,0,1,0,0,0,0,0,0,1,0,28,13.4,0.0,0.0,1.3,0,3,0,0,...,3.0,3.00,49.200000,49.200000,0.000000,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,90.0,3,1200,1140,1260,1,2,0.0,Goalkeeper
2,1,25,12,2,2025-08-31 15:30:00,1,0,3,False,90,0,0,0,1,0,0,0,0,0,2,0,12,20.0,10.0,0.0,3.0,0,12,0,0,...,1.5,1.50,31.300000,31.300000,0.000000,0.000000,0.0,0.0,2.828427,2.828427,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.954594,0.954594,8.0,180.0,4,1215,1170,1260,0,4,0.0,Goalkeeper
3,1,31,16,6,2025-09-13 11:30:00,3,0,4,False,90,0,0,1,0,0,0,0,0,0,1,0,24,12.8,0.0,0.0,1.3,0,9,0,0,...,1.0,1.00,27.533333,27.533333,3.333333,3.333333,0.0,0.0,4.000000,4.000000,0.0,0.0,0.0,0.0,0.011547,0.011547,0.011547,0.011547,0.700595,0.700595,6.0,270.0,3,1145,1120,1170,1,2,0.0,Goalkeeper
4,1,41,13,2,2025-09-21 15:30:00,1,1,5,False,90,0,0,0,1,0,0,0,0,0,2,0,13,21.4,0.0,0.0,2.1,1,6,0,0,...,0.0,0.75,15.400000,23.850000,3.333333,2.500000,0.0,0.0,2.309401,3.265986,0.0,0.0,0.0,0.0,0.011547,0.010000,0.011547,0.010000,0.193993,0.631843,6.0,360.0,4,1350,1300,1400,1,4,0.0,Goalkeeper
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23824,822,299,6,0,2026-03-14 15:00:00,0,1,30,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,...,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN,3,1195,1160,1230,1,3,0.0,Midfielder
23825,822,308,15,0,2026-03-22 12:00:00,1,2,31,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,...,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,3,1150,1150,1150,0,4,0.0,Midfielder
23826,823,306,11,0,2026-03-21 20:00:00,0,0,31,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,...,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN,3,1090,1050,1130,0,3,0.0,Defender
23827,824,306,11,0,2026-03-21 20:00:00,0,0,31,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,...,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN,3,1090,1050,1130,0,3,0.0,Defender


In [18]:
null_counts = df[FEATURE_COLS + [TARGET]].isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)

null_summary = pd.DataFrame({'nulls': null_counts, 'pct': null_pct})
null_summary = null_summary[null_summary['nulls'] > 0].sort_values('pct', ascending=False)

print(f'Features with nulls: {len(null_summary)} / {len(FEATURE_COLS)}')
print(f'Total rows: {len(df):,}\n')
null_summary

Features with nulls: 44 / 52
Total rows: 23,829



,nulls,pct
season_total_minutes,825,3.5
season_avg_pts,825,3.5
pts_rolling_3,2,0.0
pts_rolling_5,1,0.0
goals_rolling_3,2,0.0
goals_rolling_5,1,0.0
assists_rolling_3,2,0.0
assists_rolling_5,1,0.0
xg_rolling_3,2,0.0
xg_rolling_5,1,0.0


In [19]:
# Drop rows where ANY rolling feature is NaN
# This removes early-season rows where we don't have enough history
rows_before = len(df)
df_clean = df.dropna(subset=FEATURE_COLS).copy()
rows_after = len(df_clean)

print(f'Dropped {rows_before - rows_after:,} rows with NaN features ({(rows_before - rows_after)/rows_before:.1%})')
print(f'Remaining: {rows_after:,} rows')

Dropped 826 rows with NaN features (3.5%)
Remaining: 23,003 rows


In [20]:
null_counts = df_clean[FEATURE_COLS + [TARGET]].isnull().sum()
null_pct = (null_counts / len(df_clean) * 100).round(1)

null_summary = pd.DataFrame({'nulls': null_counts, 'pct': null_pct})
null_summary = null_summary[null_summary['nulls'] > 0].sort_values('pct', ascending=False)

print(f'Features with nulls: {len(null_summary)} / {len(FEATURE_COLS)}')
print(f'Total rows: {len(df_clean):,}\n')
null_summary

Features with nulls: 0 / 52
Total rows: 23,003



,nulls,pct


## 12. Spot-Check: Verify Rolling Averages

Pick a known player and manually verify the rolling calculations are correct.

In [21]:
# Pick the player with the most rows for a good demonstration
top_player = df_clean.groupby('element').size().idxmax()
player_name = players.loc[players['ID'] == top_player, 'Second Name'].iloc[0]

check = df_clean[df_clean['element'] == top_player][[
    'element', 'round', 'total_points', 'pts_rolling_3', 'pts_rolling_5',
    'season_avg_pts'
]].head(10)

print(f'Spot-check: {player_name} (element={top_player})')
print('Verify: pts_rolling_3 for row N should be mean of total_points from rows N-3 to N-1')
print()
check

Spot-check: Arrizabalaga Revuelta (element=2)
Verify: pts_rolling_3 for row N should be mean of total_points from rows N-3 to N-1



,element,round,total_points,pts_rolling_3,pts_rolling_5,season_avg_pts
32,2,2,0,3.5,3.25,0.0
33,2,3,0,0.0,2.50,0.0
34,2,4,0,0.0,1.75,0.0
35,2,5,0,0.0,0.00,0.0
36,2,6,0,0.0,0.00,0.0
37,2,7,0,0.0,0.00,0.0
38,2,8,0,0.0,0.00,0.0
39,2,9,0,0.0,0.00,0.0
40,2,10,0,0.0,0.00,0.0
41,2,11,0,0.0,0.00,0.0


## 13. Split by Position and Save to Parquet

Positions score fundamentally differently:
- **GK**: clean sheets, saves, penalty saves
- **DEF**: clean sheets, goals (from set pieces)
- **MID**: goals, assists, clean sheets (for some)
- **FWD**: goals, assists only

Separate models let each position learn its own patterns.

In [22]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'prediction', 'data')

positions = {'Goalkeeper': 'gk', 'Defender': 'def', 'Midfielder': 'mid', 'Forward': 'fwd'}

for pos_name, pos_short in positions.items():
    pos_df = df_clean[df_clean['Position'] == pos_name].copy()
    
    filepath = os.path.join(DATA_DIR, f'features_{pos_short}.parquet')
    pos_df.to_parquet(filepath, index=False)
    
    print(f'{pos_name:12s} | {pos_df.shape[0]:5,} rows | {pos_df.shape[1]:3d} cols | saved to features_{pos_short}.parquet')

# Also save the full dataset and the feature column list
df_clean.to_parquet(os.path.join(DATA_DIR, 'features_all.parquet'), index=False)

import json
with open(os.path.join(DATA_DIR, 'feature_columns.json'), 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)

print(f'\nAll positions  | {df_clean.shape[0]:5,} rows | saved to features_all.parquet')
print(f'Feature list   | {len(FEATURE_COLS)} features | saved to feature_columns.json')

Goalkeeper   | 2,649 rows |  92 cols | saved to features_gk.parquet
Defender     | 7,555 rows |  92 cols | saved to features_def.parquet
Midfielder   | 10,273 rows |  92 cols | saved to features_mid.parquet
Forward      | 2,526 rows |  92 cols | saved to features_fwd.parquet

All positions  | 23,003 rows | saved to features_all.parquet
Feature list   | 52 features | saved to feature_columns.json
